# Scop3P

A comprehensive database of human phosphosites within their full context. Scop3P integrates sequences (UniProtKB/Swiss-Prot), structures (PDB), and uniformly reprocessed phosphoproteomics data (PRIDE) to annotate all known human phosphosites. 

Scop3P, available at https://iomics.ugent.be/scop3p, presents a unique resource for visualization and analysis of phosphosites and for understanding of phosphosite structure–function relationships.

Please cite: https://doi.org/10.1021/acs.jproteome.0c00306

# Scop3P-Biophysical prediction & mutation effects 

This notebook renders analysis in 3 different tabs:

1. **WT prediction** (interactive Bokeh plot + Scop3P PTM circles)  
2. **Mutant prediction** (WT overlay in light shades + mutant in dark shades + PTM circles)  
3. **Inference** (simple summary around mutation positions)

> notes
> - Predictions are done using b2btools (DynaMine, DisOmine, EFoldMine)
 > - This app keeps the predictions in memory and the Inference tab will summarize the effect


In [29]:
# ============================================================
# Scop3P / UniProt PTM + Bio2Byte (b2bTools) prediction app
#  - Tab 1: WT prediction (PTMs from Scop3P, fallback UniProt/EBI)
#  - Tab 2: Mutant prediction (WT vs Mut plot + merged table)
#  - Tab 3: Inference (label-shift summary)
#
# Improvements included:
#  - PTM fallback for non-human proteins (UniProt/EBI features)
#  - Protein metadata (name, organism, length)
#  - Scrollable tables + download CSV/TSV links
#  - Export WT report as standalone HTML (button next to summary)
#  - Fix: Bokeh rendering (load JS once)
#  - Fix: Mutant TSV download extension
#  - Fix: remove unused status widget + old setter calls
# ============================================================

import tempfile
import requests
import pandas as pd

import ipywidgets as W
from IPython.display import display, HTML, Markdown

# --- b2bTools ---
from b2bTools import SingleSeq
try:
    from b2bTools import constants
except Exception:
    constants = None

# --- Bokeh ---
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.embed import components, file_html
from bokeh.resources import CDN, INLINE
import re


# --- Export helpers ---
import base64
from pathlib import Path
from uuid import uuid4


# ============================================================
# 1) Fetchers: UniProt sequence, metadata, PTMs (Scop3P + fallback)
# ============================================================

def fetch_uniprot_sequence(accession: str) -> str:
    """Fetch FASTA from UniProt and return the AA sequence."""
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    lines = r.text.splitlines()
    return "".join([ln.strip() for ln in lines if ln and not ln.startswith(">")])


def fetch_uniprot_metadata(accession: str) -> dict:
    """
    Fetch UniProt JSON metadata:
      - protein name (recommended → submission → alternative fallback)
      - length
      - organism (scientific/common) + taxid
    """
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
    r = requests.get(url, timeout=30, headers={"accept": "application/json"})
    r.raise_for_status()
    j = r.json()

    pdsc = j.get("proteinDescription", {}) or {}

    def _full_value(name_obj):
        if not isinstance(name_obj, dict):
            return None
        if isinstance(name_obj.get("fullName"), dict):
            return name_obj["fullName"].get("value")
        return name_obj.get("value")

    def _first_name(list_obj):
        if not isinstance(list_obj, list) or not list_obj:
            return None
        return _full_value(list_obj[0])

    protein_name = (
        _full_value(pdsc.get("recommendedName") or {}) or
        _first_name(pdsc.get("submissionNames")) or
        _first_name(pdsc.get("alternativeNames"))
    )

    # Last-resort scan
    if not protein_name:
        try:
            for key in ["recommendedName", "submissionNames", "alternativeNames", "contains"]:
                v = pdsc.get(key)
                if isinstance(v, dict):
                    protein_name = _full_value(v) or protein_name
                elif isinstance(v, list):
                    for item in v:
                        protein_name = _full_value(item) or protein_name
                        if protein_name:
                            break
                if protein_name:
                    break
        except Exception:
            pass

    # Length
    length = None
    try:
        length = int((j.get("sequence", {}) or {}).get("length"))
    except Exception:
        length = None

    # Organism
    organism = None
    taxid = None
    try:
        org = j.get("organism", {}) or {}
        organism = org.get("scientificName") or org.get("commonName")
        taxid = org.get("taxonId")
    except Exception:
        pass

    return {
        "accession": accession,
        "protein_name": protein_name,
        "length": length,
        "organism": organism,
        "taxid": taxid,
    }


def fetch_scop3p_modifications(accession: str) -> pd.DataFrame:
    """Primary PTM fetcher: Scop3P modifications endpoint."""
    url = "https://iomics.ugent.be/scop3p/api/modifications"
    r = requests.get(url, params={"accession": accession},
                     headers={"accept": "application/json"}, timeout=30)
    r.raise_for_status()
    payload = r.json()

    # payload might be a dict OR a list with one dict
    if isinstance(payload, list):
        payload = payload[0] if payload else {}

    mods = payload.get("modifications", [])
    df = pd.DataFrame(mods)

    if df.empty:
        return df

    keep = [c for c in ["position", "residue", "name", "source", "evidence", "reference", "functionalScore"]
            if c in df.columns]
    df = df[keep].copy()

    df["position"] = pd.to_numeric(df["position"], errors="coerce")
    df = df.dropna(subset=["position"])
    df["position"] = df["position"].astype(int)
    return df


def fetch_uniprot_ptm_features(accession: str, sequence: str = None) -> pd.DataFrame:
    """
    Fallback PTM fetcher using EBI Proteins API features endpoint.
    Example:
      https://www.ebi.ac.uk/proteins/api/features/P07949?categories=PTM&types=MOD_RES

    Returns a Scop3P-like table with:
      position, residue, name, source, evidence, reference
    """
    url = f"https://www.ebi.ac.uk/proteins/api/features/{accession}"
    params = {"categories": "PTM", "types": "MOD_RES"}
    r = requests.get(url, params=params, timeout=30, headers={"accept": "application/json"})
    r.raise_for_status()
    j = r.json()

    feats = j.get("features", []) if isinstance(j, dict) else []
    rows = []
    for f in feats:
        if f.get("type") != "MOD_RES":
            continue

        desc = f.get("description") or ""
        begin = f.get("begin")

        try:
            pos = int(begin)
        except Exception:
            continue

        # residue from sequence if provided
        res = ""
        if sequence and 1 <= pos <= len(sequence):
            res = sequence[pos - 1]

        eco_codes = []
        pmids = []
        for ev in (f.get("evidences") or []):
            code = ev.get("code")
            if code:
                eco_codes.append(code)
            src = (ev.get("source") or {})
            if (src.get("name") or "").lower() == "pubmed" and src.get("id"):
                pmids.append(str(src.get("id")))

        rows.append({
            "position": pos,
            "residue": res,
            "name": desc,
            "source": "UniProt",
            "evidence": ";".join(sorted(set(eco_codes))) if eco_codes else "",
            "reference": ";".join(sorted(set(pmids))) if pmids else "",
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df["position"] = pd.to_numeric(df["position"], errors="coerce")
    df = df.dropna(subset=["position"])
    df["position"] = df["position"].astype(int)
    df = df.sort_values("position").reset_index(drop=True)
    return df


def fetch_ptms_with_fallback(accession: str, sequence: str) -> tuple[pd.DataFrame, str]:
    """
    1) Try Scop3P PTMs
    2) If empty/fails → UniProt/EBI PTM features
    Returns:
      (mods_df, ptm_source_label) where label is "Scop3P", "UniProt", or "None"
    """
    try:
        mods = fetch_scop3p_modifications(accession)
        if mods is not None and not mods.empty:
            mods = mods.copy()
            if "source" not in mods.columns:
                mods["source"] = "Scop3P"
            return mods, "Scop3P"
    except Exception:
        pass

    try:
        mods_u = fetch_uniprot_ptm_features(accession, sequence=sequence)
        if mods_u is not None and not mods_u.empty:
            return mods_u, "UniProt"
    except Exception:
        pass

    return pd.DataFrame(columns=["position", "residue", "name", "source", "evidence", "reference"]), "None"


# ============================================================
# 1.1) Parse pasted FASTA/sequence (confidential custom input)
# ============================================================

def parse_fasta_or_sequence(text: str) -> tuple[str, str]:
    """
    Accepts FASTA (with header) or raw AA sequence.
    Returns (seq, header_label).
    """
    if text is None:
        raise ValueError("No sequence provided.")

    raw = text.strip()
    if not raw:
        raise ValueError("Please paste a FASTA or amino-acid sequence.")

    header = "CUSTOM_SEQ"
    lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]

    if lines and lines[0].startswith(">"):
        header = lines[0][1:].strip() or "CUSTOM_SEQ"
        seq_lines = [ln for ln in lines[1:] if not ln.startswith(">")]
    else:
        seq_lines = lines

    seq = "".join(seq_lines).replace(" ", "").replace("\t", "").upper()

    allowed = set("ACDEFGHIKLMNPQRSTVWYX")
    bad = sorted(set([c for c in seq if c not in allowed]))
    if bad:
        raise ValueError(f"Sequence contains invalid characters: {bad[:20]}")

    if len(seq) < 5:
        raise ValueError("Sequence is too short.")

    return seq, header


# ============================================================
# 2) Export helpers: download links (CSV/TSV) + HTML report export
# ============================================================

def download_link_df(df: pd.DataFrame, filename: str, sep: str = ",",
                     link_text: str = "Download", width: str = "200px", height: str = "38px",
                     bg: str = "#17a2b8", border: str = "#138496", text: str = "#fff"):
    if df is None or df.empty:
        return HTML("<i>No data to download.</i>")

    text_csv = df.to_csv(index=False, sep=sep)
    b64 = base64.b64encode(text_csv.encode("utf-8")).decode("utf-8")
    mime = "text/tab-separated-values" if sep == "\t" else "text/csv"

    html = f"""
    <a download="{filename}"
       href="data:{mime};charset=utf-8;base64,{b64}"
       style="
         display:inline-flex;
         align-items:center;
         justify-content:center;
         width:{width};
         height:{height};
         padding:0 10px;
         border-radius:6px;
         text-decoration:none;
         color:{text};
         background:{bg};
         border:1px solid {border};
         font-weight:600;
         box-sizing:border-box;
       ">
       {link_text}
    </a>
    """
    return HTML(html)





def export_report_html(
    acc: str,
    prot_name: str,
    organism: str,
    length: int,
    ptm_src: str,
    fig,
    tables: dict,
    out_path: str = None,
):
    """
    Writes a single self-contained HTML report (Bokeh JS INLINE) with:
      - protein summary
      - interactive Bokeh plot
      - one or more tables (scrollable wrappers)
    """
    if out_path is None:
        out_path = f"{acc}_report.html"

    # Standalone Bokeh embed (no string replacement hacks)
    script, div = components(fig)
    resources = INLINE.render()

    # Basic styling so huge tables are usable
    css = """
    <style>
      body { font-family: Arial, sans-serif; margin: 16px; }
      .summary { margin: 10px 0; padding: 10px; border: 1px solid #ddd; border-radius: 8px; }
      .tbl-title { font-weight: 700; margin: 16px 0 6px 0; }
      .tbl-wrap { width: 100%; max-width: 100%; overflow: auto; border: 1px solid #ddd;
                  border-radius: 8px; padding: 6px; }
      table.dataframe { border-collapse: collapse; font-size: 12px; width: max-content; min-width: 100%; }
      table.dataframe th, table.dataframe td { border: 1px solid #eee; padding: 6px 8px; white-space: nowrap; }
      table.dataframe thead th { position: sticky; top: 0; background: #f7f7f7; z-index: 5; }
    </style>
    """

    summary_html = f"""
    <div class="summary">
      <div><b>Protein name:</b> {prot_name}</div>
      <div><b>UniProt:</b> {acc}</div>
      <div><b>Organism:</b> {organism}</div>
      <div><b>Length:</b> {length}</div>
      <div><b>PTMs source:</b> {ptm_src}</div>
    </div>
    """

    table_blocks = []
    for title, df in (tables or {}).items():
        if df is None or df.empty:
            continue
        table_blocks.append(f"<div class='tbl-title'>{title}</div>")
        table_blocks.append("<div class='tbl-wrap'>")
        table_blocks.append(df.to_html(index=False, escape=True, classes="dataframe"))
        table_blocks.append("</div>")

    html_doc = f"""<!doctype html>
    <html>
    <head>
      <meta charset="utf-8"/>
      <title>{acc} report</title>
      {resources}
      {css}
    </head>
    <body>
      {summary_html}
      {div}
      {script}
      {''.join(table_blocks)}
    </body>
    </html>
    """

    Path(out_path).write_text(html_doc, encoding="utf-8")
    return out_path

# ============================================================
# 3) Prediction: b2bTools → dataframe + mutation utility
# ============================================================

def predict_biophysical(accession: str, sequence: str) -> dict:
    """Run b2bTools SingleSeq prediction and return the raw dict."""
    with tempfile.NamedTemporaryFile(prefix="seq_", suffix=".fasta", mode="w") as fp:
        fp.write(f">{accession}\n{sequence}\n")
        fp.flush()
        fp.seek(0)

        s = SingleSeq(fp.name)

        if constants is not None:
            tool_candidates = []
            for name in ["TOOL_BACKBONE_DYNAMICS", "TOOL_DYNAMINE", "TOOL_DISOMINE", "TOOL_EFOLDMINE"]:
                if hasattr(constants, name):
                    tool_candidates.append(getattr(constants, name))

            pred = s.predict(tools=tool_candidates).get_all_predictions() if tool_candidates else s.predict().get_all_predictions()
        else:
            pred = s.predict().get_all_predictions()

    return pred


def prediction_to_df(pred: dict, accession: str) -> pd.DataFrame:
    """Convert b2bTools output to a dataframe."""
    prot = None
    if isinstance(pred, dict):
        if "proteins" in pred and accession in pred["proteins"]:
            prot = pred["proteins"][accession]
        elif accession in pred:
            prot = pred[accession]

    if prot is None:
        raise ValueError("Could not find protein predictions for requested accession in b2bTools output.")

    n = len(prot.get("seq", "")) or len(prot.get("backbone", [])) or len(prot.get("disoMine", [])) or len(prot.get("earlyFolding", []))
    df = pd.DataFrame({
        "seqpos": list(range(1, n+1)),
        "backbone": prot.get("backbone", [None]*n),
        "disoMine": prot.get("disoMine", [None]*n),
        "earlyFolding": prot.get("earlyFolding", [None]*n),
    })
    for c in ["backbone", "disoMine", "earlyFolding"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def apply_mutations(sequence: str, positions_csv: str, aas_csv: str) -> str:
    """Apply 1-indexed mutations: positions '10,25' and aas 'A,V'."""
    pos = [p.strip() for p in positions_csv.split(",") if p.strip()]
    aas = [a.strip().upper() for a in aas_csv.split(",") if a.strip()]
    if len(pos) != len(aas):
        raise ValueError("Positions and amino acids must have the same count (e.g., '10,25' and 'A,V').")

    seq = list(sequence)
    for p_str, aa in zip(pos, aas):
        p = int(p_str)
        if p < 1 or p > len(seq):
            raise ValueError(f"Position {p} out of range (sequence length {len(seq)}).")
        if len(aa) != 1 or aa not in set("ACDEFGHIKLMNPQRSTVWY"):
            raise ValueError(f"Invalid amino acid '{aa}' at position {p}.")
        seq[p-1] = aa
    return "".join(seq)


# ============================================================
# 4) Bokeh plotting (load BokehJS once for Voila stability)
# ============================================================


_BOKEH_JS_LOADED = False

def _embed_bokeh(fig) -> None:
    global _BOKEH_JS_LOADED
    script, div = components(fig)

    # INLINE = no async CDN load race
    if not _BOKEH_JS_LOADED:
        display(HTML(INLINE.render()))
        _BOKEH_JS_LOADED = True

    # components() returns a <script>...</script> block; we wrap it to wait for Bokeh
    js = re.sub(r"^<script[^>]*>|</script>$", "", script.strip())

    display(HTML(div + f"""
    <script>
    (function() {{
      function renderWhenReady() {{
        if (window.Bokeh && window.Bokeh.embed) {{
          {js}
          setTimeout(() => window.dispatchEvent(new Event('resize')), 100);
        }} else {{
          setTimeout(renderWhenReady, 50);
        }}
      }}
      renderWhenReady();
    }})();
    </script>
    """))


def make_wt_plot(wt_df: pd.DataFrame, mods_df: pd.DataFrame, title_text: str):
    p = figure(
        width=1000, height=300,
        tools="pan,box_zoom,reset,save",
        toolbar_location="below", toolbar_sticky=False
    )
    p.title.text = title_text

    l1 = p.line(wt_df["seqpos"], wt_df["backbone"], line_width=2, color="blue", alpha=0.8,
                muted_color="blue", muted_alpha=0.2, legend_label="backbone_dynamics")
    l2 = p.line(wt_df["seqpos"], wt_df["disoMine"], line_width=2, color="red", alpha=0.8,
                muted_color="red", muted_alpha=0.2, legend_label="disorder")
    l3 = p.line(wt_df["seqpos"], wt_df["earlyFolding"], line_width=2, color="grey", alpha=0.8,
                muted_color="grey", muted_alpha=0.2, legend_label="earlyFolding")

    p.add_tools(HoverTool(tooltips="Seqpos:@x, value:@y", renderers=[l1, l2, l3]))

    # PTM markers
    if mods_df is not None and not mods_df.empty and "position" in mods_df.columns:
        ptm_src = ColumnDataSource(dict(
            x=mods_df["position"].tolist(),
            y=[0.5]*len(mods_df),
            residue=mods_df.get("residue", pd.Series([""]*len(mods_df))).astype(str).tolist(),
            name=mods_df.get("name", pd.Series([""]*len(mods_df))).astype(str).tolist(),
            source=mods_df.get("source", pd.Series([""]*len(mods_df))).astype(str).tolist(),
        ))
        ptm_renderer = p.scatter(
            x="x", y="y", source=ptm_src,
            marker="circle", size=10,
            fill_alpha=0.6, line_alpha=0.8,
            color="grey", legend_label="P-sites"
        )
        p.add_tools(HoverTool(
            tooltips=[("Seqpos", "@x"), ("residue", "@residue"), ("mod", "@name"), ("source", "@source")],
            renderers=[ptm_renderer]
        ))

    p.legend.click_policy = "mute"
    p.add_layout(p.legend[0], "right")
    return p


def make_mut_plot(wt_df: pd.DataFrame, mut_df: pd.DataFrame, mods_df: pd.DataFrame, title_text: str):
    p = figure(
        width=1000, height=300,
        tools="pan,box_zoom,reset,save",
        toolbar_location="below", toolbar_sticky=False
    )
    p.title.text = title_text

    # WT (lighter)
    b1 = p.line(wt_df["seqpos"], wt_df["backbone"], line_width=2, color="skyblue", alpha=0.8,
                muted_color="skyblue", muted_alpha=0.2, legend_label="backbone (WT)")
    d1 = p.line(wt_df["seqpos"], wt_df["disoMine"], line_width=2, color="salmon", alpha=0.8,
                muted_color="salmon", muted_alpha=0.2, legend_label="disorder (WT)")
    e1 = p.line(wt_df["seqpos"], wt_df["earlyFolding"], line_width=2, color="grey", alpha=0.8,
                muted_color="grey", muted_alpha=0.2, legend_label="earlyFolding (WT)")

    # Mut (darker)
    b2 = p.line(mut_df["seqpos"], mut_df["backbone"], line_width=2, color="blue", alpha=0.8,
                muted_color="blue", muted_alpha=0.2, legend_label="backbone (Mut)")
    d2 = p.line(mut_df["seqpos"], mut_df["disoMine"], line_width=2, color="red", alpha=0.8,
                muted_color="red", muted_alpha=0.2, legend_label="disorder (Mut)")
    e2 = p.line(mut_df["seqpos"], mut_df["earlyFolding"], line_width=2, color="black", alpha=0.8,
                muted_color="black", muted_alpha=0.2, legend_label="earlyFolding (Mut)")

    p.add_tools(HoverTool(tooltips="Seqpos:@x, value:@y", renderers=[b1, b2, d1, d2, e1, e2]))

    # PTM markers
    if mods_df is not None and not mods_df.empty and "position" in mods_df.columns:
        ptm_src = ColumnDataSource(dict(
            x=mods_df["position"].tolist(),
            y=[0.5]*len(mods_df),
            residue=mods_df.get("residue", pd.Series([""]*len(mods_df))).astype(str).tolist(),
            name=mods_df.get("name", pd.Series([""]*len(mods_df))).astype(str).tolist(),
            source=mods_df.get("source", pd.Series([""]*len(mods_df))).astype(str).tolist(),
        ))
        ptm_renderer = p.scatter(
            x="x", y="y", source=ptm_src,
            marker="circle", size=10,
            fill_alpha=0.6, line_alpha=0.8,
            color="grey", legend_label="P-sites"
        )
        p.add_tools(HoverTool(
            tooltips=[("Seqpos", "@x"), ("residue", "@residue"), ("mod", "@name"), ("source", "@source")],
            renderers=[ptm_renderer]
        ))

    p.legend.click_policy = "mute"
    p.add_layout(p.legend[0], "right")
    return p


# ============================================================
# 5) Inference helpers: class labels + shift tables
# ============================================================

def label_backbone(x: float) -> str:
    if pd.isna(x): return "NA"
    if x > 1.0:  return "membrane-spanning"
    if x > 0.8:  return "rigid"
    if x > 0.69: return "context-dependent"
    return "flexible"

def label_disorder(x: float) -> str:
    if pd.isna(x): return "NA"
    return "disordered" if x > 0.50 else "ordered"

def label_earlyfold(x: float) -> str:
    if pd.isna(x): return "NA"
    return "early-folding" if x > 0.169 else "non-early-folding"

LABEL_FUNCS = {
    "backbone": (label_backbone, "Backbone dynamics"),
    "disoMine": (label_disorder, "Disorder"),
    "earlyFolding": (label_earlyfold, "Early folding"),
}

def parse_mutations(pos_str: str, aa_str: str):
    pos = [p.strip() for p in pos_str.split(",") if p.strip()]
    aa  = [a.strip().upper() for a in aa_str.split(",") if a.strip()]
    if len(pos) != len(aa):
        raise ValueError("Number of positions must match number of amino acids.")
    muts = []
    for p, a in zip(pos, aa):
        if not p.isdigit():
            raise ValueError(f"Invalid position '{p}'.")
        p_i = int(p)
        if len(a) != 1 or a not in set("ACDEFGHIKLMNPQRSTVWY"):
            raise ValueError(f"Invalid amino acid '{a}' at position {p_i}.")
        muts.append((p_i, a))
    return muts


def mutation_effect_table_with_label_shift(wt_df: pd.DataFrame,
                                          mut_df: pd.DataFrame,
                                          feature: str,
                                          mutations,
                                          window: int = 5) -> pd.DataFrame:
    """Mutation-centric inference table: label changes at site and window mean."""
    if feature not in LABEL_FUNCS:
        raise ValueError(f"feature must be one of {list(LABEL_FUNCS.keys())}")

    label_fn, pretty = LABEL_FUNCS[feature]

    wt = wt_df.set_index("seqpos", drop=False)
    mu = mut_df.set_index("seqpos", drop=False)
    max_pos = int(wt["seqpos"].max())

    rows = []
    for pos, aa_to in mutations:
        if pos not in wt.index or pos not in mu.index:
            continue

        wt_center = float(wt.loc[pos, feature])
        mu_center = float(mu.loc[pos, feature])
        d_center  = mu_center - wt_center

        lo = max(1, pos - window)
        hi = min(max_pos, pos + window)

        wt_mean = float(wt.loc[lo:hi, feature].astype(float).mean())
        mu_mean = float(mu.loc[lo:hi, feature].astype(float).mean())
        d_mean = mu_mean - wt_mean

        wt_lab_pos = label_fn(wt_center)
        mu_lab_pos = label_fn(mu_center)
        wt_lab_mean = label_fn(wt_mean)
        mu_lab_mean = label_fn(mu_mean)

        shift_pos = f"{wt_lab_pos} → {mu_lab_pos}" if wt_lab_pos != mu_lab_pos else f"{wt_lab_pos} (no class change)"
        shift_mean = f"{wt_lab_mean} → {mu_lab_mean}" if wt_lab_mean != mu_lab_mean else f"{wt_lab_mean} (no class change)"

        wt_aa = str(wt.loc[pos, "seq"])

        rows.append({
            "pos": pos,
            "WT_AA": wt_aa,
            "Mut_AA": aa_to,
            "mutation": f"{wt_aa}{pos}{aa_to}",
            f"{feature}_WT@pos": wt_center,
            f"{feature}_Mut@pos": mu_center,
            "Δ@pos": d_center,
            f"{feature}_WT_mean": wt_mean,
            f"{feature}_Mut_mean": mu_mean,
            "Δ_mean": d_mean,
            "label_shift@pos": shift_pos,
            "label_shift_mean": shift_mean,
            "inference": (
                f"{pretty}: {shift_pos} at site (Δ {d_center:+.3f}); "
                f"window mean: {shift_mean} (Δ {d_mean:+.3f})"
            ),
        })

    df = pd.DataFrame(rows)
    num_cols = [c for c in df.columns if any(k in c for k in ["_WT@pos", "_Mut@pos", "_WT_mean", "_Mut_mean", "Δ@pos", "Δ_mean"])]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").round(3)
    return df


# ============================================================
# 6) Table helpers: PTM flag + scrollable display
# ============================================================

def _non_runtime_pred_cols(df: pd.DataFrame):
    """Return prediction columns excluding runtime-like columns and core columns."""
    core = {"seqpos", "seq"}
    cols = []
    for c in df.columns:
        if c in core:
            continue
        if "runtime" in str(c).lower():
            continue
        cols.append(c)
    return cols

def _add_ptm_flag(df: pd.DataFrame, mods_df: pd.DataFrame) -> pd.DataFrame:
    """Add PTMs yes/no based on positions."""
    out = df.copy()
    ptm_positions = set()
    if mods_df is not None and not mods_df.empty and "position" in mods_df.columns:
        ptm_positions = set(pd.to_numeric(mods_df["position"], errors="coerce").dropna().astype(int).tolist())
    out["PTMs"] = out["seqpos"].apply(lambda x: "yes" if int(x) in ptm_positions else "no")
    return out

def make_wt_table(wt_df: pd.DataFrame, mods_df: pd.DataFrame) -> pd.DataFrame:
    cols = _non_runtime_pred_cols(wt_df)
    base = wt_df[["seqpos", "seq"] + cols].copy()
    return _add_ptm_flag(base, mods_df)

def make_wt_mut_merged_table(wt_df: pd.DataFrame, mut_df: pd.DataFrame, mods_df: pd.DataFrame) -> pd.DataFrame:
    wt_cols = _non_runtime_pred_cols(wt_df)
    mut_cols = _non_runtime_pred_cols(mut_df)
    common = [c for c in wt_cols if c in mut_cols]

    w = wt_df[["seqpos", "seq"] + common].copy().rename(columns={"seq": "WT_AA"})
    m = mut_df[["seqpos", "seq"] + common].copy().rename(columns={"seq": "Mut_AA"})
    merged = pd.merge(w, m, on="seqpos", how="inner", suffixes=("_WT", "_Mut"))

    ordered = ["seqpos", "WT_AA", "Mut_AA"]
    for c in common:
        ordered += [f"{c}_WT", f"{c}_Mut"]
    merged = merged[ordered]
    return _add_ptm_flag(merged, mods_df)


def display_track_guide():
    display(HTML("""
<div style="margin:6px 0 10px 0; padding:8px 10px; border:1px solid #ddd; border-radius:8px;">
  <div style="margin-bottom:6px;"><b>How to read the tracks</b></div>
  <div style="margin:2px 0;">
    <span style="color:blue; font-weight:600;">Backbone dynamics</span>:
    &gt;1.0 membrane-spanning, 0.8–1.0 rigid, 0.69–0.80 context-dependent, &lt;0.69 flexible
  </div>
  <div style="margin:2px 0;">
    <span style="color:red; font-weight:600;">Disorder (DisoMine)</span>:
    values &gt;0.50 indicate disordered regions
  </div>
  <div style="margin:2px 0;">
    <span style="color:grey; font-weight:600;">Early folding</span>:
    values &gt;0.169 suggest early-folding propensity
  </div>
  <div style="margin:2px 0;">
    <span style="color:grey; font-weight:600;">P-sites</span>:
    phosphorylation positions (grey dots)
  </div>
</div>
"""))


def display_scrollable_table(
    df: pd.DataFrame,
    *,
    title: str = "",
    height_px: int = 420,
    width: str = "100%",
    sticky_cols: int = 0,
    col_widths_px=None,
    highlight_seqpos=None,
    seqpos_col: str = "seqpos",
):
    """Voila-safe scrollable HTML table with sticky header and optional row highlighting."""
    if df is None or df.empty:
        display(HTML(f"<div><b>{title}</b><br><i>(no rows)</i></div>" if title else "<i>(no rows)</i>"))
        return

    table_id = f"tbl_{uuid4().hex}"

    if col_widths_px is None:
        col_widths_px = [90] * sticky_cols
    else:
        col_widths_px = (list(col_widths_px) + [90] * sticky_cols)[:sticky_cols]

    left_offsets, acc = [], 0
    for w in col_widths_px:
        left_offsets.append(acc)
        acc += int(w)

    hl = set(int(x) for x in highlight_seqpos) if highlight_seqpos else set()

    if hl and seqpos_col in df.columns:
        cols = list(df.columns)
        thead = "<thead><tr>" + "".join([f"<th>{c}</th>" for c in cols]) + "</tr></thead>"
        rows_html = []
        for _, r in df.iterrows():
            try:
                pos_val = int(r[seqpos_col])
            except Exception:
                pos_val = None
            cls = " class='row-hl'" if (pos_val is not None and pos_val in hl) else ""
            tds = "".join([f"<td>{'' if pd.isna(r[c]) else str(r[c])}</td>" for c in cols])
            rows_html.append(f"<tr{cls}>{tds}</tr>")
        tbody = "<tbody>" + "".join(rows_html) + "</tbody>"
        html_table = f"<table border='1' class='dataframe'>{thead}{tbody}</table>"
    else:
        html_table = df.to_html(index=False, escape=True)

    sticky_css_cols = []
    for i in range(1, sticky_cols + 1):
        left = left_offsets[i - 1]
        z = 4 + (sticky_cols - i)
        sticky_css_cols.append(f"""
#{table_id} table th:nth-child({i}),
#{table_id} table td:nth-child({i}) {{
  position: sticky;
  left: {left}px;
  z-index: {z};
  background: #fff;
}}
""")

    css = f"""
<style>
#{table_id} .tbl-title {{
  font-weight: 700;
  margin: 8px 0 6px 0;
}}

#{table_id} tr.row-hl td {{
  background: #fff6cc;
}}

#{table_id} .tbl-wrap {{
  width: {width};
  max-width: {width};
  height: {height_px}px;
  overflow: auto;
  border: 1px solid #ddd;
  border-radius: 8px;
}}

#{table_id} table {{
  border-collapse: collapse;
  width: max-content;
  min-width: 100%;
  font-size: 12px;
}}

#{table_id} th, #{table_id} td {{
  border: 1px solid #eee;
  padding: 6px 8px;
  white-space: nowrap;
}}

#{table_id} thead th {{
  position: sticky;
  top: 0;
  z-index: 10;
  background: #f7f7f7;
}}

{''.join(sticky_css_cols)}
</style>
"""

    block = f"""
<div id="{table_id}">
  {f'<div class="tbl-title">{title}</div>' if title else ''}
  <div class="tbl-wrap">
    {html_table}
  </div>
</div>
"""
    display(HTML(css + block))


# ============================================================
# 7) UI + App State + Status widgets
# ============================================================

# --- WT input mode: UniProt or pasted FASTA/sequence ---
wt_mode = W.ToggleButtons(
    options=[("UniProt ID", "uniprot"), ("Paste FASTA/Seq", "paste")],
    value="uniprot",
    description="",
)


acc_in = W.Text(value="P07949", description="UniProt:", layout=W.Layout(width="260px"))

seq_in = W.Textarea(
    value="",
    description="FASTA/Seq:",
    placeholder="Paste FASTA (with >header) OR raw amino-acid sequence here…",
    layout=W.Layout(width="720px", height="120px"),
)

seq_hint = W.HTML(
    "<div style='font-size:12px;color:#666;margin-left:6px;'>"
    "Tip: you can paste either a FASTA block (starting with &gt;) or a plain AA sequence."
    "</div>"
)

def _toggle_wt_inputs(change=None):
    is_unip = (wt_mode.value == "uniprot")
    acc_in.layout.display = "" if is_unip else "none"
    seq_in.layout.display = "none" if is_unip else ""
    seq_hint.layout.display = "none" if is_unip else ""

_toggle_wt_inputs()
wt_mode.observe(_toggle_wt_inputs, names="value")

run_wt_btn = W.Button(description="Fetch & Predict WT", button_style="success")

wt_out = W.Output()

pos_in = W.Text(value="606", description="Positions:", placeholder="e.g. 10,25,100", layout=W.Layout(width="320px"))
aa_in  = W.Text(value="A", description="To AA:", placeholder="e.g. A,V,G", layout=W.Layout(width="320px"))
run_mut_btn = W.Button(description="Apply & Predict", button_style="warning", layout=W.Layout(width="320px"))
mut_out = W.Output()

run_inf_btn = W.Button(description="Run inference", button_style="info")
inf_out = W.Output()

APP_STATE = {
    "accession": None,
    "seq_label": None,
    "mode": None,
    "sequence": None,
    "mods_df": None,
    "wt_df": None,
    "mut_df": None,
    "mutations": None,
    "meta": None,
    "ptm_src": None,
}

status_wt  = W.HTML(value="")
status_mut = W.HTML(value="")
status_inf = W.HTML(value="")

def _set_status_wt(msg: str):
    status_wt.value = f"<div style='padding:6px;border:1px solid #ddd;border-radius:6px;'>{msg}</div>"

def _set_status_mut(msg: str):
    status_mut.value = f"<div style='padding:6px;border:1px solid #ddd;border-radius:6px;'>{msg}</div>"

def _set_status_inf(msg: str):
    status_inf.value = f"<div style='padding:6px;border:1px solid #ddd;border-radius:6px;'>{msg}</div>"


# ============================================================
# 8) Callbacks: WT / Mutant / Inference
# ============================================================

def do_wt(_=None):
    BTN_W, BTN_H = "200px","38px" 
    """
    Tab 1:
      - Fetch UniProt sequence + metadata
      - Fetch PTMs (Scop3P first, fallback to UniProt features)
      - Run b2bTools predictions (WT)
      - Show protein summary + "Export WT HTML" button
      - Show Bokeh plot
      - Show WT table with CSV/TSV download links above it
    """
    with wt_out:
        wt_out.clear_output()
        try:
            # -------------------------
            # 1) Inputs + fetches
            # -------------------------
            acc = acc_in.value.strip()
            _set_status_wt("Fetching UniProt sequence + protein metadata…")

            # --- WT input mode switch (UniProt vs pasted FASTA/sequence) ---
            if wt_mode.value == "uniprot":
                seq_label = acc
                seq  = fetch_uniprot_sequence(acc)
                meta = fetch_uniprot_metadata(acc)

                _set_status_wt("Fetching PTMs (Scop3P → fallback UniProt if needed)…")
                mods, ptm_src = fetch_ptms_with_fallback(acc, seq)

            else:
                # Paste FASTA/Seq mode
                seq, header = parse_fasta_or_sequence(seq_in.value)
                seq_label = header
                meta = {
                    "accession": "CUSTOM",
                    "protein_name": header or "Custom sequence",
                    "length": len(seq),
                    "organism": "(user-provided sequence)",
                    "taxid": None,
                }
                mods = pd.DataFrame(columns=["position", "residue", "name", "source", "evidence", "reference"])
                ptm_src = "None"
                acc = "CUSTOM"
                _set_status_wt("Using pasted FASTA/sequence (no PTMs fetch).")

            # -------------------------
            # 2) Predictions (WT)
            # -------------------------
            _set_status_wt("Running WT biophysical prediction…")
            pred = predict_biophysical(seq_label, seq)
            wt_df = prediction_to_df(pred, seq_label)
            wt_df["seq"] = list(seq)

            # cache state for other tabs (mutant/inference)
            APP_STATE.update({
                "accession": acc,
                "seq_label": seq_label,
                "mode": wt_mode.value,
                "sequence": seq,
                "mods_df": mods,
                "wt_df": wt_df,
                "mut_df": None,
                "mutations": None,
                "meta": meta,
                "ptm_src": ptm_src,
            })

            # -------------------------
            # 3) Prepare summary strings
            # -------------------------
            prot_name = meta.get("protein_name") or "(name not available)"
            length    = meta.get("length") or len(seq)
            organism  = meta.get("organism") or "(organism not available)"
            taxid     = meta.get("taxid")

            unip_link = f"https://www.uniprot.org/uniprotkb/{acc}/entry"
            tax_txt   = f" (taxid {taxid})" if taxid else ""
            ptm_note  = " (no PTMs found)" if ptm_src == "None" else ""

            # Build WT figure early so we can export it too
            fig = make_wt_plot(
                wt_df, mods,
                title_text=f"Biophysical properties (WT) — {acc} | {prot_name}"
            )

            # -------------------------
            # 4) Protein summary + HTML export (WT)
            # -------------------------
            if wt_mode.value == "uniprot":
                summary_html = f"""
                <div style="margin:6px 0 10px 0; padding:8px 10px; border:1px solid #ddd; border-radius:8px;">
                  <div><b>Protein name:</b> {prot_name}</div>
                  <div><b>UniProt_ID:</b> <a href="{unip_link}" target="_blank">{acc}</a></div>
                  <div><b>Organism:</b> {organism}{tax_txt}</div>
                  <div><b>Length:</b> {length}</div>
                  <div><b>PTMs source:</b> {ptm_src}{ptm_note}</div>
                </div>
                """
            else:
                summary_html = f"""
                <div style="margin:6px 0 10px 0; padding:8px 10px; border:1px solid #ddd; border-radius:8px;">
                  <div><b>Protein name:</b> {prot_name}</div>
                  <div><b>Input:</b> pasted FASTA/sequence</div>
                  <div><b>Organism:</b> {organism}</div>
                  <div><b>Length:</b> {length}</div>
                  <div><b>PTMs source:</b> {ptm_src}{ptm_note}</div>
                </div>
                """

            btn_export_wt = W.Button(description="Export WT HTML", button_style="info",layout=W.Layout(width=BTN_W,height=BTN_H))
            export_wt_out = W.Output()

            def _export_wt_html(_btn=None):
                # Use the same table shown on screen (WT)
                wt_tbl_local = make_wt_table(wt_df, mods)

                out_path = export_report_html(
                    acc=acc,
                    prot_name=prot_name,
                    organism=organism,
                    length=length,
                    ptm_src=ptm_src,
                    fig=fig,
                    tables={"WT predicted features (per residue)": wt_tbl_local},
                    out_path=f"{acc}_WT_report.html"
                )
                with export_wt_out:
                    export_wt_out.clear_output()
                    display(HTML(f"<div style='margin-top:6px;'><b>Saved:</b> {out_path}</div>"))

            btn_export_wt.on_click(_export_wt_html)

            display(W.HTML(summary_html))


            # -------------------------
            # 5) Plot
            # -------------------------
            _set_status_wt(f"WT prediction ready. (PTMs: {0 if mods is None else len(mods)})")

            display_track_guide()
            _embed_bokeh(fig)

            # -------------------------
            # 6) WT table + downloads
            # -------------------------
            # --- WT table + downloads/export above it ---
            display(HTML("<hr style='margin:10px 0;'>"))
            wt_tbl = make_wt_table(wt_df, mods)
            

            btn_export_wt.layout = W.Layout(width=BTN_W,height=BTN_H)
            
            dl_row = W.HBox([
                W.HTML(download_link_df(wt_tbl, f"{acc}_WT_table.csv", sep=",",
                                        link_text="Download WT (CSV)", width=BTN_W,height=BTN_H).data),
                W.HTML(download_link_df(wt_tbl, f"{acc}_WT_table.tsv", sep="\t",
                                        link_text="Download WT (TSV)", width=BTN_W,height=BTN_H).data),
                btn_export_wt
            ])
            display(dl_row)
            display(export_wt_out)


            display_scrollable_table(
                wt_tbl,
                title="WT predicted features (per residue)",
                height_px=420,
                width="100%",
                sticky_cols=0,
            )

        except Exception as e:
            _set_status_wt(f"<b>Error:</b> {e}")
            raise


def do_mut(_=None):
    BTN_W, BTN_H = "200px","38px" 
    """
    Tab 2:
      - Apply user mutations to WT sequence
      - Run b2bTools predictions on mutant
      - Show mutation labels, Bokeh WT vs Mut plot
      - Show merged WT vs Mut table with CSV/TSV downloads
      - Add "Export WT vs Mut HTML" (plot + merged table)
    """
    with mut_out:
        mut_out.clear_output()
        try:
            # -------------------------
            # 1) Preconditions
            # -------------------------
            if APP_STATE.get("wt_df") is None or APP_STATE.get("sequence") is None:
                raise ValueError("Run WT prediction first.")

            acc    = APP_STATE["accession"]
            seq    = APP_STATE["sequence"]
            mods   = APP_STATE["mods_df"]
            meta   = APP_STATE.get("meta") or {}
            ptm_src = APP_STATE.get("ptm_src") or "Scop3P/UniProt"

            prot_name = meta.get("protein_name") or "(protein name unknown)"
            organism  = meta.get("organism") or "(organism not available)"
            length    = meta.get("length") or len(seq)

            # -------------------------
            # 2) Mutate + predict
            # -------------------------
            _set_status_mut("Applying mutations + predicting…")

            # build mutated sequence
            mut_seq = apply_mutations(seq, pos_in.value, aa_in.value)

            # predict mutant
            # NOTE: for pasted sequences we still use the same accession label as WT for b2bTools keying
            seq_label = APP_STATE.get("seq_label") or acc
            pred_mut = predict_biophysical(seq_label, mut_seq)
            mut_df = prediction_to_df(pred_mut, seq_label)
            mut_df["seq"] = list(mut_seq)

            APP_STATE.update({"mut_df": mut_df, "mutations": (pos_in.value, aa_in.value)})

            _set_status_mut("Mutant prediction ready.")

            # -------------------------
            # 3) Mutation summary (WT_AAposMutAA)
            # -------------------------
            muts = parse_mutations(pos_in.value, aa_in.value)
            labels = []
            for p, aa_to in muts:
                wt_aa = seq[p-1] if 1 <= p <= len(seq) else "?"
                labels.append(f"{wt_aa}{p}{aa_to}")
            msg = ", ".join(labels) if labels else "(none)"

            display(HTML(
                f"<div style='margin:6px 0 10px 0;'><b>Predicted properties for mutants:</b> {msg}</div>"
            ))

            # -------------------------
            # 4) Plot (WT vs Mut)
            # -------------------------
            fig = make_mut_plot(
                APP_STATE["wt_df"],
                mut_df,
                mods,
                title_text=f"Biophysical properties (WT vs Mutant) — {acc} | {prot_name}"
            )

            display_track_guide()
            _embed_bokeh(fig)

            # -------------------------
            # 5) Merged table + downloads + HTML export button
            # -------------------------
            display(HTML("<hr style='margin:10px 0;'>"))

            merged_tbl = make_wt_mut_merged_table(APP_STATE["wt_df"], mut_df, mods)
            highlight_positions = [p for p, _ in muts]

            # Export button (plot + merged table)
            btn_export_mut = W.Button(description="Export WT vs Mut HTML", button_style="info",layout=W.Layout(width=BTN_W,height=BTN_H))
            export_mut_out = W.Output()

            def _export_mut_html(_btn=None):
                out_path = export_report_html(
                    acc=acc,
                    prot_name=prot_name,
                    organism=organism,
                    length=length,
                    ptm_src=ptm_src,
                    fig=fig,
                    tables={"WT vs Mutant predicted features (aligned by seqpos)": merged_tbl},
                    out_path=f"{acc}_WT_vs_MUT_report.html"
                )
                with export_mut_out:
                    export_mut_out.clear_output()
                    display(HTML(f"<div style='margin-top:6px;'><b>Saved:</b> {out_path}</div>"))

            btn_export_mut.on_click(_export_mut_html)

            # Downloads above table (plus export HTML button)


            btn_export_mut.layout = W.Layout(width=BTN_W,height=BTN_H)
            
            dl_row = W.HBox([
                W.HTML(download_link_df(merged_tbl, f"{acc}_WT_vs_MUT_table.csv", sep=",",
                                        link_text="Download WT vs Mut (CSV)", width=BTN_W,height=BTN_H).data),
                W.HTML(download_link_df(merged_tbl, f"{acc}_WT_vs_MUT_table.tsv", sep="\t",
                                        link_text="Download WT vs Mut (TSV)", width=BTN_W,height=BTN_H).data),
                btn_export_mut
            ])
            display(dl_row)
            display(export_mut_out)


            display_scrollable_table(
                merged_tbl,
                title="WT vs Mutant predicted features (aligned by seqpos)",
                height_px=420,
                width="100%",
                sticky_cols=0,
                highlight_seqpos=highlight_positions,
                seqpos_col="seqpos",
            )

        except Exception as e:
            _set_status_mut(f"<b>Error:</b> {e}")
            raise

def do_inf(_=None):
    with inf_out:
        inf_out.clear_output()
        try:
            if APP_STATE["wt_df"] is None or APP_STATE["mut_df"] is None:
                raise ValueError("Run WT prediction and Mutant prediction first.")

            wt_df = APP_STATE["wt_df"]
            mut_df = APP_STATE["mut_df"]
            muts = parse_mutations(pos_in.value, aa_in.value)

            _set_status_inf("Running inference…")

            display(Markdown("## Label-shift summary (±5 AA window)"))
            for feat, (_, pretty) in LABEL_FUNCS.items():
                display(Markdown(f"### {pretty}"))
                df = mutation_effect_table_with_label_shift(
                    wt_df=wt_df,
                    mut_df=mut_df,
                    feature=feat,
                    mutations=muts,
                    window=5
                )
                display(df)

            _set_status_inf("Inference ready.")
        except Exception as e:
            _set_status_inf(f"<b>Error:</b> {e}")
            raise


# ============================================================
# 9) Wire callbacks + layout tabs
# ============================================================

run_wt_btn.on_click(do_wt)
run_mut_btn.on_click(do_mut)
run_inf_btn.on_click(do_inf)


wt_controls = W.VBox([
    W.HBox([wt_mode, run_wt_btn]),
    acc_in,
    seq_in,
    seq_hint
])

wt_box  = W.VBox([wt_controls, status_wt, wt_out])
mut_box = W.VBox([W.HBox([pos_in, aa_in, run_mut_btn]), status_mut, mut_out])
inf_box = W.VBox([status_inf, run_inf_btn, inf_out])

tabs = W.Tab(children=[wt_box, mut_box, inf_box])
tabs.set_title(0, "WT prediction")
tabs.set_title(1, "Mutant prediction")
tabs.set_title(2, "Inference")

display(tabs)

def _on_tab_change(change):
    display(HTML("<script>setTimeout(()=>window.dispatchEvent(new Event('resize')), 100);</script>"))

tabs.observe(_on_tab_change, names="selected_index")
